# Arabic Speech Processing Pipeline
### Run each cell independently — no need to re-run previous stages

| Cell | Stage | Needs Internet | Time |
|---|---|---|---|
| 1 | Imports | ❌ | seconds |
| 2 | ASR - Whisper | ✅ | ~60-90 mins |
| 3 | Summarization - mT5 | ❌ | ~20-30 mins |
| 4 | Search Index - E5 | ❌ | ~5 mins |
| 5 | Keyword Spotting | ❌ | seconds |
| 6 | Search Demo | ❌ | seconds |

In [ ]:
# ── CELL 1: IMPORTS ──────────────────────────────────────────
# Run this first, always

from datasets import load_dataset
import pandas as pd
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import os

# Make sure outputs folder exists
os.makedirs('outputs', exist_ok=True)

print('✅ All imports successful')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# ── CELL 2: STAGE 1 - ASR ────────────────────────────────────
# Run ONCE only — saves to outputs/asr_results.csv
# Skip this cell if asr_results.csv already exists

import os
if os.path.exists('outputs/asr_results.csv'):
    print('✅ asr_results.csv already exists — skipping ASR stage')
    print('   Delete outputs/asr_results.csv to re-run this stage')
else:
    print('[STAGE 1] Loading ASR model (Whisper)...')
    asr = pipeline(
        'automatic-speech-recognition',
        model='openai/whisper-small',
        generate_kwargs={
            'language': 'arabic',
            'task': 'transcribe',
            'no_repeat_ngram_size': 3
        },
        return_timestamps=True
    )

    dataset = load_dataset('hirundo-io/MASC', split='train', streaming=True)
    asr_results = []
    TARGET = 100

    print(f'[STAGE 1] Transcribing {TARGET} audio samples...')
    for i, sample in enumerate(dataset):
        if len(asr_results) >= TARGET:
            break

        filename = sample['audio']['path'].split('/')[-1]

        try:
            audio_array = sample['audio']['array'][:60 * sample['audio']['sampling_rate']]
            transcript = asr({
                'array': audio_array,
                'sampling_rate': sample['audio']['sampling_rate']
            })['text'].strip()

            print(f'  [{len(asr_results)+1}/{TARGET}] {filename} → {transcript[:60]}...')
            asr_results.append({'filename': filename, 'transcript': transcript})

        except Exception as e:
            print(f'  [SKIP] {filename} — {str(e)[:60]}')
            continue

    asr_df = pd.DataFrame(asr_results)
    asr_df.to_csv('outputs/asr_results.csv', index=False, encoding='utf-8-sig')
    print(f'[STAGE 1] ✅ Done. Saved {len(asr_df)} transcripts.')

In [ ]:
# ── CELL 3: STAGE 2 - SUMMARIZATION ──────────────────────────
# Run ONCE only — saves to outputs/summarization_results.csv
# Skip this cell if summarization_results.csv already exists

import os
if os.path.exists('outputs/summarization_results.csv'):
    print('✅ summarization_results.csv already exists — skipping summarization stage')
    print('   Delete outputs/summarization_results.csv to re-run this stage')
else:
    # Load ASR results
    asr_df = pd.read_csv('outputs/asr_results.csv')
    print(f'Loaded {len(asr_df)} transcripts')

    print('[STAGE 2] Loading Summarization model (mT5)...')
    tokenizer = AutoTokenizer.from_pretrained('csebuetnlp/mT5_multilingual_XLSum')
    sum_model = AutoModelForSeq2SeqLM.from_pretrained('csebuetnlp/mT5_multilingual_XLSum')

    sum_results = []
    print('[STAGE 2] Summarizing transcripts...')
    for i, row in asr_df.iterrows():
        try:
            inputs = tokenizer(
                row['transcript'],
                return_tensors='pt',
                max_length=512,
                truncation=True
            )
            with torch.no_grad():
                output_ids = sum_model.generate(
                    inputs['input_ids'],
                    max_new_tokens=60,
                    num_beams=4,
                    early_stopping=True
                )
            summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
        except Exception as e:
            summary = f'ERROR: {e}'

        print(f'  [{i+1}/{len(asr_df)}] {row["filename"]} → {summary[:60]}...')
        sum_results.append({
            'filename': row['filename'],
            'transcript': row['transcript'],
            'summary': summary
        })

    sum_df = pd.DataFrame(sum_results)
    sum_df.to_csv('outputs/summarization_results.csv', index=False, encoding='utf-8-sig')
    print(f'[STAGE 2] ✅ Done. Saved {len(sum_df)} summaries.')

In [ ]:
# ── CELL 4: STAGE 3 - SEMANTIC SEARCH INDEX ──────────────────
# Run whenever you change the embedding model
# Fast — takes only ~5 minutes

# Load summarization results
sum_df = pd.read_csv('outputs/summarization_results.csv')
print(f'Loaded {len(sum_df)} documents')

print('[STAGE 3] Loading Embedding model (E5-base)...')
embedder = SentenceTransformer('intfloat/multilingual-e5-base')

print('[STAGE 3] Building search index...')
embeddings = embedder.encode(
    ['passage: ' + t for t in sum_df['transcript'].tolist()],
    show_progress_bar=True,
    normalize_embeddings=True
)
embeddings = np.array(embeddings).astype('float32')
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

faiss.write_index(index, 'outputs/search_index.faiss')
sum_df.to_csv('outputs/search_metadata.csv', index=False, encoding='utf-8-sig')
print(f'[STAGE 3] ✅ Done. Index built with {index.ntotal} vectors.')

In [ ]:
# ── CELL 5: STAGE 4 - KEYWORD SPOTTING ───────────────────────
# Fast — runs in seconds

asr_df = pd.read_csv('outputs/asr_results.csv')

keywords = [
    'الله',
    'كندا',
    'ألمانيا',
    'برمجة',
    'رمضان',
    'صحة',
    'تعليم',
    'اقتصاد',
    'سياسة',
    'تكنولوجيا',
]

print('=' * 60)
print('   KEYWORD SPOTTING RESULTS')
print('=' * 60)

keyword_results = []
for keyword in keywords:
    matches = asr_df[asr_df['transcript'].str.contains(keyword, na=False)]
    if len(matches) > 0:
        print(f"\n🔑 Keyword '{keyword}' found in {len(matches)} file(s):")
        for _, row in matches.iterrows():
            print(f"     📁 {row['filename']}")
            print(f"     📝 {row['transcript'][:80]}...")
            keyword_results.append({
                'keyword': keyword,
                'filename': row['filename'],
                'transcript_preview': row['transcript'][:100]
            })
    else:
        print(f"\n🔑 Keyword '{keyword}' — not found")

kw_df = pd.DataFrame(keyword_results)
if not kw_df.empty:
    kw_df.to_csv('outputs/keyword_spotting_results.csv', index=False, encoding='utf-8-sig')
    print(f'\n[STAGE 4] ✅ Done. Found {len(keyword_results)} keyword matches.')
else:
    print('\n[STAGE 4] ✅ Done. No matches found.')

In [ ]:
# ── CELL 6: SEARCH DEMO ──────────────────────────────────────
# Run this cell any time to test search queries
# Change the queries list below to test different searches

# Load saved data
sum_df = pd.read_csv('outputs/search_metadata.csv')
embedder = SentenceTransformer('intfloat/multilingual-e5-base')

# Rebuild index
embeddings = embedder.encode(
    ['passage: ' + t for t in sum_df['transcript'].tolist()],
    show_progress_bar=True,
    normalize_embeddings=True
)
embeddings = np.array(embeddings).astype('float32')
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

# ── Define your test queries here ────────────────────────────
test_queries = [
    'ألمانيا وهامبورغ',
    'كندا واستراليا',
    'صحة وغذاء',
    'برمجة وتصميم',
    'دين وإسلام',
]

print('=' * 60)
print('   SEARCH RESULTS')
print('=' * 60)

for query in test_queries:
    query_vec = embedder.encode(
        ['query: ' + query],
        normalize_embeddings=True
    ).astype('float32')

    distances, indices = index.search(query_vec, 3)

    print(f"\nQuery: '{query}'")
    print('-' * 50)
    for rank, idx in enumerate(indices[0]):
        score = distances[0][rank]
        if score > 0.6:
            relevance = '✅ Relevant'
        elif score > 0.4:
            relevance = '🟡 Possible match'
        else:
            relevance = '⚠️ Weak match'

        print(f'  Rank {rank+1}: {relevance}')
        print(f"  📁 File    : {sum_df.iloc[idx]['filename']}")
        print(f"  📝 Summary : {sum_df.iloc[idx]['summary']}")
        print(f'  📊 Score   : {score:.4f}  (1.0 = perfect match)')
        print()

In [ ]:
# ── CELL 7: EVALUATION - Precision@K ─────────────────────────
# Run after Cell 6 to measure search quality

# Known correct answers — update these after checking your results
test_queries_eval = [
    ('ألمانيا وهامبورغ',   '-SuGpbd7KMI.wav'),
    ('كندا واستراليا',      '-kudo6VQZwE.wav'),
    ('صحة وغذاء وخضروات',  '1yPxazSDrwk.wav'),
    ('برمجة وتصميم',       '6dX8TbyW9Cg.wav'),
    ('رمضان وأغاني',       '-sVkr0x8Rrg.wav'),
]

correct_at_1 = 0
correct_at_3 = 0

print('=' * 60)
print('   EVALUATION RESULTS — Precision@K')
print('=' * 60)

for query, expected_file in test_queries_eval:
    query_vec = embedder.encode(
        ['query: ' + query],
        normalize_embeddings=True
    ).astype('float32')

    distances, indices = index.search(query_vec, 3)
    retrieved_files = [sum_df.iloc[idx]['filename'] for idx in indices[0]]

    hit_at_1 = retrieved_files[0] == expected_file
    hit_at_3 = expected_file in retrieved_files

    if hit_at_1: correct_at_1 += 1
    if hit_at_3: correct_at_3 += 1

    print(f'\nQuery   : {query}')
    print(f'Expected: {expected_file}')
    print(f'Got     : {retrieved_files[0]}')
    print(f'P@1     : {"✅ Correct" if hit_at_1 else "❌ Wrong"}')
    print(f'P@3     : {"✅ In top 3" if hit_at_3 else "❌ Not found"}')

print('\n' + '=' * 60)
print(f'Precision@1 : {correct_at_1}/{len(test_queries_eval)} = {correct_at_1/len(test_queries_eval)*100:.0f}%')
print(f'Precision@3 : {correct_at_3}/{len(test_queries_eval)} = {correct_at_3/len(test_queries_eval)*100:.0f}%')
print('=' * 60)